# Resume NER v7 — Production Ready (FIXED)
### التحسينات من v6 → v7:
1. ✅ **Designation: يطلع واحد بس** — أقصر span + header priority + skills matching
2. ✅ **Degree: يشيل section headers + GPA + date ranges**
3. ✅ **Val/Test Gap Fix** — رفع النسبة لـ 15% val + 15% test
4. ✅ **Synthetic data: Designation training**
5. ✅ **Post-processing: entity dedup logic**
6. ✅ **Companies: شيل usernames و emails و urls**

In [1]:
# Cell 1: Install
!pip install -q transformers datasets evaluate seqeval matplotlib peft accelerate pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 16.1 MB/s eta 0:00:00


In [2]:
# Cell 2: Imports & Seed
import os, re, json, copy, random
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter, deque
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
    get_cosine_schedule_with_warmup,
)
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from pypdf import PdfReader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [3]:
# Cell 3: Configuration
MODEL_NAME   = 'dslim/bert-base-NER'
DATA_PATH    = 'Entity Recognition in Resumes.json'
OUTPUT_DIR   = './resume_ner_v7_lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LEN      = 256
BATCH_SIZE   = 16
GRAD_ACCUM   = 2
EPOCHS       = 15

LR           = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.1
LORA_MODULES = ['query', 'key', 'value', 'dense']

WEIGHT_EXP   = 0.40
MAX_WEIGHT   = 5.0
LABEL_SMOOTH = 0.05

N_AUGMENT    = 3
N_SYNTHETIC  = 300

PATIENCE     = 3
SMOOTH_WIN   = 2

TTA_RUNS     = 3
INF_CONF     = 0.50

VAL_RATIO    = 0.15
TEST_RATIO   = 0.15

print('Config loaded.')

Config loaded.


In [4]:
# Cell 4: قاموس Skills الضخم (200+ skill)
SKILLS_DICT = {
    "Programming Languages": [
        "Python", "Java", "JavaScript", "TypeScript", "C", "C++", "C#",
        "Go", "Rust", "Swift", "Kotlin", "R", "MATLAB", "Scala", "Ruby",
        "PHP", "Perl", "Shell", "Bash", "PowerShell", "Dart", "Lua",
    ],
    "Web Frontend": [
        "React", "Vue.js", "Angular", "Next.js", "Svelte",
        "HTML", "CSS", "Sass", "Tailwind CSS", "Bootstrap",
        "Material UI", "Redux", "GraphQL", "REST API",
        "WebSockets", "jQuery", "Webpack", "Vite",
    ],
    "Web Backend": [
        "Node.js", "Express.js", "Django", "Flask", "FastAPI",
        "Spring Boot", "Laravel", "Ruby on Rails",
        "ASP.NET", "NestJS", "Microservices", "gRPC", "OAuth", "JWT",
    ],
    "Databases": [
        "PostgreSQL", "MySQL", "SQLite", "Oracle", "SQL Server",
        "MongoDB", "Redis", "Cassandra", "Elasticsearch", "DynamoDB",
        "Firebase", "Neo4j", "Snowflake", "BigQuery", "Redshift",
        "MariaDB", "SQL",
    ],
    "Cloud & DevOps": [
        "AWS", "Azure", "Google Cloud", "GCP", "Docker", "Kubernetes",
        "Terraform", "Ansible", "Jenkins", "GitHub Actions",
        "Helm", "Prometheus", "Grafana", "Nginx", "Linux",
        "Serverless", "Lambda", "EC2", "S3", "RDS",
    ],
    "AI & Machine Learning": [
        "TensorFlow", "PyTorch", "Scikit-learn", "Keras", "Hugging Face",
        "XGBoost", "LightGBM", "CatBoost",
        "Machine Learning", "Deep Learning", "Neural Networks",
        "Natural Language Processing", "NLP", "Computer Vision",
        "Reinforcement Learning", "Generative AI",
        "Feature Engineering", "Model Evaluation",
    ],
    "Data Science & Analytics": [
        "Pandas", "NumPy", "SciPy", "Matplotlib", "Seaborn", "Plotly",
        "Tableau", "Power BI", "Looker", "Jupyter",
        "Data Analysis", "Data Visualization", "Statistical Analysis",
        "A/B Testing", "ETL", "Data Pipeline", "Apache Spark",
        "Data Cleaning", "Data Transformation", "Data Modeling",
        "EDA", "DAX", "Power Query",
    ],
    "Mobile": [
        "Android", "iOS", "React Native", "Flutter", "SwiftUI",
    ],
    "Software Engineering": [
        "Git", "GitHub", "GitLab", "Agile", "Scrum",
        "JIRA", "TDD", "Unit Testing", "CI/CD", "System Design",
    ],
    "Other Tools": [
        "Excel", "Word", "PowerPoint", "Figma",
        "Postman", "Swagger",
    ],
}

ALL_SKILLS_FLAT  = [s for cat in SKILLS_DICT.values() for s in cat]
SKILLS_LOWER_MAP = {s.lower(): s for s in ALL_SKILLS_FLAT}
SKILLS_POOL      = ALL_SKILLS_FLAT.copy()

DESIGNATION_LIST = [
    'Software Engineer', 'Data Scientist', 'Backend Developer',
    'Full Stack Developer', 'ML Engineer', 'DevOps Engineer',
    'Frontend Developer', 'Data Analyst', 'Cloud Architect',
    'NLP Engineer', 'Computer Vision Engineer', 'AI Researcher',
    'Site Reliability Engineer', 'Data Engineer', 'Product Manager',
    'Business Analyst', 'Systems Analyst', 'Database Administrator',
    'Security Engineer', 'Machine Learning Engineer',
    'Data Analyst and Machine Learning Engineer',
    'Software Developer', 'Web Developer', 'Mobile Developer',
    'QA Engineer', 'Test Engineer', 'Platform Engineer',
]
DESIG_LOWER_MAP = {d.lower(): d for d in DESIGNATION_LIST}

SECTION_HEADERS = {
    'education', 'experience', 'experiences', 'skills', 'summary',
    'objective', 'projects', 'certifications', 'languages', 'awards',
    'publications', 'references', 'contact', 'profile', 'about',
    'technical skills', 'soft skills', 'work experience',
    'professional experience', 'academic background',
}

print(f'✅ Skills: {len(ALL_SKILLS_FLAT)} | Designations: {len(DESIGNATION_LIST)}')

✅ Skills: 153 | Designations: 27


In [7]:
# Cell 5: Load Data + Build Label Schema
data = []
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line: data.append(json.loads(line))

print(f'Total resumes: {len(data)}')

raw_labels = set()
for item in data:
    for ann in item.get('annotation', []):
        if ann.get('label'):
            lbl = ann['label'][0]
            if 'UNKNOWN' not in lbl: raw_labels.add(lbl)

ENTITIES   = sorted(raw_labels)
ALL_LABELS = ['O'] + [f'{p}-{e}' for e in ENTITIES for p in ('B', 'I')]
label2id   = {l: i for i, l in enumerate(ALL_LABELS)}
id2label   = {i: l for i, l in enumerate(ALL_LABELS)}
N_LABELS   = len(ALL_LABELS)

print(f'Entities ({len(ENTITIES)}): {ENTITIES}')
print(f'Total labels: {N_LABELS}')

Total resumes: 220
Entities (10): ['College Name', 'Companies worked at', 'Degree', 'Designation', 'Email Address', 'Graduation Year', 'Location', 'Name', 'Skills', 'Years of Experience']
Total labels: 21


In [8]:
# Cell 6: Parser
def clean_text(t):
    if not isinstance(t, str): return ''
    return t.encode('utf-8', errors='ignore').decode('utf-8')

def parse_resume(content, annotations):
    content  = clean_text(content)
    char_ent = {}
    for ann in (annotations or []):
        if not ann or not ann.get('label') or not ann.get('points'): continue
        lbl = ann['label'][0]
        if 'UNKNOWN' in lbl or lbl not in raw_labels: continue
        for pt in ann['points']:
            for c in range(pt.get('start', 0), pt.get('end', 0) + 1):
                char_ent[c] = lbl

    words, ws, we = [], [], []
    for m in re.finditer(r'\S+', content):
        words.append(m.group()); ws.append(m.start()); we.append(m.end())
    if not words: return None

    bio, prev = [], None
    for s, e in zip(ws, we):
        hits = [char_ent[c] for c in range(s, e) if c in char_ent]
        if hits:
            ent = Counter(hits).most_common(1)[0][0]
            bio.append(label2id[f"{'I' if ent==prev else 'B'}-{ent}"])
            prev = ent
        else:
            bio.append(label2id['O']); prev = None
    return [clean_text(w) for w in words if clean_text(w)], bio

all_tokens, all_tags = [], []
for row in data:
    r = parse_resume(row.get('content',''), row.get('annotation',[]))
    if r and len(r[0]) == len(r[1]):
        all_tokens.append(r[0]); all_tags.append(r[1])

print(f'Parsed resumes: {len(all_tokens)}')

Parsed resumes: 220


In [9]:
# Cell 7: Data Augmentation
def augment_sample(tokens, tags):
    augmented = []
    for _ in range(N_AUGMENT):
        new_tokens, new_tags = [], []
        for tok, tag_id in zip(tokens, tags):
            lbl = id2label[tag_id]
            if lbl == 'O' and random.random() < 0.05: continue
            elif lbl == 'O' and random.random() < 0.10:
                new_tokens.append(tok.lower()); new_tags.append(tag_id)
            else:
                new_tokens.append(tok); new_tags.append(tag_id)
        if len(new_tokens) >= 5:
            augmented.append((new_tokens, new_tags))
    return augmented

aug_tokens, aug_tags = [], []
for toks, tags in zip(all_tokens, all_tags):
    for at, ag in augment_sample(toks, tags):
        aug_tokens.append(at); aug_tags.append(ag)

print(f'Augmented samples: {len(aug_tokens)}')

Augmented samples: 660


In [10]:
# Cell 8: Synthetic Data
NAMES = ['Ahmed Ali','Sara Mohamed','Omar Hassan','Fatima Khalid',
         'Youssef Ibrahim','Nour Ahmed','Aditi Sharma','Rahul Kumar',
         'John Smith','Emily Johnson','Michael Brown','Priya Patel',
         'Khaled Mansour','Laila Hassan','Mostafa Samir','Rana Tarek',
         'David Wilson','Sophia Martinez','Mohamed Hammad','Aisha Diallo']

COMPANIES = ['Google','Amazon','Microsoft','IBM','Oracle','Infosys',
             'TCS','Wipro','Accenture','Meta','Apple','Netflix',
             'Spotify','Uber','Airbnb','Stripe','Shopify',
             'Vodafone','Etisalat','DEPI','Careem','Noon']

DEGREES   = ['B.Tech','M.Tech','B.E.','M.S.','MBA','B.Sc','M.Sc','Ph.D',
             "Bachelor's", "Master's", "Bachelor's in Computer Science",
             "Bachelor's in Artificial Intelligence",
             "Bachelor's in Data Science", "Master's in AI",
             "Bachelor's in Information Technology"]

FIELDS    = ['Computer Science','Information Technology',
             'Data Science','Artificial Intelligence',
             'Software Engineering','Electronics','Mathematics']

COLLEGES  = ['IIT Delhi','IIT Bombay','NIT Trichy','BITS Pilani',
             'Cairo University','Ain Shams University','MIT',
             'Stanford University','Alexandria University',
             'Helwan University','AUB','KAUST']

LOCATIONS = ['Cairo','Alexandria','Bangalore','Mumbai','Delhi',
             'Dubai','Abu Dhabi','Riyadh','London','Berlin','Toronto',
             'Cairo, Egypt','Dubai, UAE','Bangalore, India']

JOB_BULLETS = {
    'Data Analyst': [
        'Developed interactive dashboards using Power BI and Tableau.',
        'Processed and cleaned data using SQL and Python.',
        'Performed EDA to uncover business insights.',
        'Built reports with DAX measures for KPI tracking.',
    ],
    'Machine Learning Engineer': [
        'Built end-to-end ML pipelines for classification tasks.',
        'Applied Feature Engineering and Model Evaluation techniques.',
        'Deployed models using Docker and AWS.',
        'Improved model accuracy through hyperparameter tuning.',
    ],
    'Software Engineer': [
        'Developed RESTful APIs using Django and FastAPI.',
        'Managed CI/CD pipelines with GitHub Actions.',
        'Wrote unit tests and performed code reviews.',
        'Deployed applications using Docker and Kubernetes.',
    ],
    'Data Scientist': [
        'Built predictive models using Scikit-learn and XGBoost.',
        'Performed statistical analysis and A/B testing.',
        'Visualized insights using Matplotlib and Seaborn.',
        'Developed NLP pipelines for text classification.',
    ],
    'Backend Developer': [
        'Designed microservices architecture with Node.js.',
        'Optimized database queries in PostgreSQL.',
        'Implemented authentication with JWT and OAuth.',
        'Deployed services on AWS Lambda.',
    ],
}
DEFAULT_BULLETS = [
    'Collaborated with cross-functional teams.',
    'Delivered projects on time following Agile methodology.',
    'Documented technical specifications and system design.',
]

def make_synthetic():
    name    = random.choice(NAMES)
    desg    = random.choice(DESIGNATION_LIST[:15])
    company = random.choice(COMPANIES)
    degree_phrase = random.choice(DEGREES)
    field   = random.choice(FIELDS)
    college = random.choice(COLLEGES)
    loc     = random.choice(LOCATIONS)
    parts   = name.lower().split()
    email   = f"{parts[0]}.{parts[-1]}@gmail.com"

    skills   = random.sample(SKILLS_POOL, random.randint(4, 7))
    sk_text  = '\n'.join(f"- {s}" for s in skills)

    bullets = JOB_BULLETS.get(desg, DEFAULT_BULLETS)
    exp_text = '\n'.join(f"• {b}" for b in random.sample(bullets, min(2, len(bullets))))

    text = (
        f"{name}\n"
        f"{desg}\n"
        f"{email} | {loc}\n\n"
        f"SUMMARY\n"
        f"Experienced {desg} with strong background in {field}.\n\n"
        f"EXPERIENCE\n"
        f"{company} | {desg} | 2021 - Present | {loc}\n"
        f"{exp_text}\n\n"
        f"EDUCATION\n"
        f"{degree_phrase} in {field}\n"
        f"{college} | 2019 | {loc}\n\n"
        f"SKILLS\n"
        f"{sk_text}"
    )

    words = text.split()
    bio   = [label2id['O']] * len(words)

    def tag_span(target, label):
        if f'B-{label}' not in label2id: return
        tw = target.split()
        for i in range(len(words) - len(tw) + 1):
            if words[i:i+len(tw)] == tw:
                bio[i] = label2id[f'B-{label}']
                for j in range(1, len(tw)): bio[i+j] = label2id[f'I-{label}']
                return

    tag_span(name,          'Name')
    tag_span(desg,          'Designation')
    tag_span(email,         'Email Address')
    tag_span(loc.split(',')[0].strip(), 'Location')
    tag_span(company,       'Companies worked at')
    full_degree = f"{degree_phrase} in {field}"
    tag_span(full_degree,   'Degree')
    tag_span(college,       'College Name')
    for sk in skills: tag_span(sk, 'Skills')

    return words, bio

syn_tokens, syn_tags = [], []
for _ in range(N_SYNTHETIC):
    w, t = make_synthetic()
    syn_tokens.append(w); syn_tags.append(t)

print(f'Synthetic samples: {len(syn_tokens)}')

Synthetic samples: 300


In [11]:
# Cell 9: Train/Val/Test Split
random.seed(SEED)
indices = list(range(len(all_tokens)))
random.shuffle(indices)

n_val  = max(15, int(len(indices) * VAL_RATIO))
n_test = max(15, int(len(indices) * TEST_RATIO))

val_idx   = indices[:n_val]
test_idx  = indices[n_val:n_val+n_test]
train_idx = indices[n_val+n_test:]

real_tokens = [all_tokens[i] for i in train_idx]
real_tags   = [all_tags[i]   for i in train_idx]
val_tokens  = [all_tokens[i] for i in val_idx]
val_tags    = [all_tags[i]   for i in val_idx]
test_tokens = [all_tokens[i] for i in test_idx]
test_tags   = [all_tags[i]   for i in test_idx]

train_tokens   = real_tokens + aug_tokens + syn_tokens
train_tags_all = real_tags   + aug_tags   + syn_tags

print(f'Train: {len(train_tokens):,}  (real={len(real_tokens)} aug={len(aug_tokens)} syn={len(syn_tokens)})')
print(f'Val:   {len(val_tokens):>4}  ({VAL_RATIO*100:.0f}% real)')
print(f'Test:  {len(test_tokens):>4}  ({TEST_RATIO*100:.0f}% real)')
print(f'Gap ratio train/val: {len(train_tokens)/len(val_tokens):.1f}x')

Train: 1,114  (real=154 aug=660 syn=300)
Val:     33  (15% real)
Test:    33  (15% real)
Gap ratio train/val: 33.8x


In [12]:
# Cell 10: Tokenizer + Label Alignment
print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(tokens_list, tags_list):
    return Dataset.from_dict({'tokens': tokens_list, 'ner_tags': tags_list})

dataset = DatasetDict({
    'train':      make_hf_dataset(train_tokens,   train_tags_all),
    'val':        make_hf_dataset(val_tokens,     val_tags),
    'test':       make_hf_dataset(test_tokens,    test_tags),
    'train_real': make_hf_dataset(real_tokens,    real_tags),
})

def tokenize_align(examples):
    enc = tokenizer(
        examples['tokens'], truncation=True,
        max_length=MAX_LEN, is_split_into_words=True
    )
    all_lbl = []
    for i, tags in enumerate(examples['ner_tags']):
        wids = enc.word_ids(i); lbls, prev = [], None
        for w in wids:
            if w is None:   lbls.append(-100)
            elif w != prev: lbls.append(min(int(tags[w]), N_LABELS - 1))
            else:           lbls.append(-100)
            prev = w
        all_lbl.append(lbls)
    enc['labels'] = all_lbl
    return enc

print('Tokenising ...')
tok = dataset.map(tokenize_align, batched=True, remove_columns=['tokens','ner_tags'])
print(f'Done. Train: {len(tok["train"]):,} | Val: {len(tok["val"])} | Test: {len(tok["test"])}')

Loading tokenizer ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/1114 [00:00<?, ? examples/s]

Map:   0%|          | 0/33 [00:00<?, ? examples/s]

Map:   0%|          | 0/33 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

Done. Train: 1,114 | Val: 33 | Test: 33


In [13]:
# Cell 11: Class Weights
counts = Counter()
for s in tok['train']:
    for l in s['labels']:
        if l != -100: counts[l] += 1

total = sum(counts.values())
weights = []
print(f"  {'Label':<32}  {'Count':>8}  {'Weight':>7}")
for i in range(N_LABELS):
    c = max(counts.get(i, 1), 1)
    w = min((total / (N_LABELS * c)) ** WEIGHT_EXP, MAX_WEIGHT)
    weights.append(w)
    print(f'  {ALL_LABELS[i]:<32}  {c:>8}  {w:>7.3f}')

wt = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

  Label                                Count   Weight
  O                                   106128    0.320
  B-College Name                         694    2.390
  I-College Name                        1234    1.898
  B-Companies worked at                 1506    1.753
  I-Companies worked at                  944    2.113
  B-Degree                               657    2.443
  I-Degree                              1788    1.637
  B-Designation                         1585    1.717
  I-Designation                         2710    1.386
  B-Email Address                       1041    2.032
  I-Email Address                        277    3.450
  B-Graduation Year                      286    3.407
  I-Graduation Year                        1    5.000
  B-Location                            1351    1.831
  I-Location                             105    5.000
  B-Name                                1110    1.980
  I-Name                                1163    1.944
  B-Skills                  

In [16]:
# Cell 12: Model + LoRA
base_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=N_LABELS, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
).to(DEVICE)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_MODULES,
    lora_dropout=LORA_DROPOUT, bias='none', task_type=TaskType.TOKEN_CLS,
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model.to(DEVICE)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |                                                                                      
-------------------------+------------+--------------------------------------------------------------------------------------
bert.pooler.dense.weight | UNEXPECTED |                                                                                      
bert.pooler.dense.bias   | UNEXPECTED |                                                                                      
classifier.weight        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9, 768]) vs model:torch.Size([21, 768])
classifier.bias          | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9]) vs model:torch.Size([21])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not mat

trainable params: 5,324,565 || all params: 113,060,394 || trainable%: 4.7095


PeftModelForTokenClassification(
  (base_model): LoraModel(
    (model): BertForTokenClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(28996, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(

In [17]:
# Cell 13: seqeval Metrics
seqeval = evaluate.load('seqeval')

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, -1)
    tp, tl = [], []
    for p, l in zip(preds, labels):
        tp.append([ALL_LABELS[x] for x, y in zip(p, l) if y != -100])
        tl.append([ALL_LABELS[y] for y in l if y != -100])
    r = seqeval.compute(predictions=tp, references=tl, zero_division=0)
    out = {
        'precision': r['overall_precision'], 'recall': r['overall_recall'],
        'f1': r['overall_f1'], 'accuracy': r['overall_accuracy'],
    }
    for k, v in r.items():
        if isinstance(v, dict) and 'f1' in v: out[f'{k}_f1'] = v['f1']
    return out

In [15]:
!pip uninstall -y torchao
!pip install torchao --upgrade

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.8 MB/s eta 0:00:00


In [18]:
# Cell 14: Scheduler + Training Args
steps_ep    = max(1, len(tok['train']) // (BATCH_SIZE * GRAD_ACCUM))
total_steps = steps_ep * EPOCHS
warmup      = max(1, int(WARMUP_RATIO * total_steps))

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-8
)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup, total_steps)

print(f'Scheduler: steps/ep={steps_ep} | warmup={warmup} | total={total_steps}')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=32,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
    weight_decay=WEIGHT_DECAY, warmup_steps=0, lr_scheduler_type='constant',
    eval_strategy='epoch', save_strategy='no', load_best_model_at_end=False,
    logging_steps=max(1, steps_ep // 4), report_to='none',
    fp16=True, max_grad_norm=1.0, seed=SEED,
)

Scheduler: steps/ep=34 | warmup=30 | total=510


In [19]:
# Cell 15: Custom Trainer — Composite Early Stopping (70% F1 + 30% Skills F1)
best_f1, best_skills_f1, best_state = -1.0, -1.0, None
no_improve = 0
hist = {'ep':[],'tr_loss':[],'vl_loss':[],'vl_f1':[],'tr_f1':[],
        'vl_p':[],'vl_r':[],'vl_acc':[],'vl_skills_f1':[],'lr':[]}

class NERTrainer(Trainer):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self._in_train = False
        self._f1_buf   = deque(maxlen=SMOOTH_WIN)

    def train(self, *a, **kw):
        self._in_train = True
        r = super().train(*a, **kw)
        self._in_train = False
        return r

    def create_optimizer_and_scheduler(self, num_training_steps):
        self.optimizer = optimizer; self.lr_scheduler = scheduler

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out    = model(**inputs)
        loss   = nn.CrossEntropyLoss(
            weight=wt, ignore_index=-100, label_smoothing=LABEL_SMOOTH
        )(out.logits.view(-1, N_LABELS), labels.view(-1))
        return (loss, out) if return_outputs else loss

    def evaluate(self, eval_dataset=None, *a, **kw):
        global best_f1, best_skills_f1, best_state, no_improve
        vm   = super().evaluate(eval_dataset=eval_dataset, *a, **kw)
        vf1  = vm.get('eval_f1', 0.0)
        vlss = vm.get('eval_loss', 0.0)
        vp   = vm.get('eval_precision', 0.0)
        vr   = vm.get('eval_recall', 0.0)
        vacc = vm.get('eval_accuracy', 0.0)

        skills_key    = next((k for k in vm if 'skills' in k.lower() and '_f1' in k.lower()), None)
        vskills_f1    = vm.get(skills_key, 0.0) if skills_key else 0.0

        tp  = self.predict(tok['train_real'])
        tm  = compute_metrics((tp.predictions, tp.label_ids))
        tf1 = tm['f1']

        tlss = next((e['loss'] for e in reversed(self.state.log_history)
                     if 'loss' in e and 'eval_loss' not in e), 0.0)
        lr = optimizer.param_groups[0]['lr']
        ep = int(self.state.epoch or 0)

        hist['ep'].append(ep);               hist['tr_loss'].append(tlss)
        hist['vl_loss'].append(vlss);        hist['vl_f1'].append(vf1)
        hist['tr_f1'].append(tf1);           hist['vl_p'].append(vp)
        hist['vl_r'].append(vr);             hist['vl_acc'].append(vacc)
        hist['vl_skills_f1'].append(vskills_f1); hist['lr'].append(lr)

        if not self._in_train: return vm

        self._f1_buf.append(vf1)
        sf1       = float(np.mean(self._f1_buf))
        composite = 0.70 * sf1 + 0.30 * vskills_f1

        if composite > (0.70 * best_f1 + 0.30 * best_skills_f1) + 1e-4:
            best_f1, best_skills_f1 = sf1, vskills_f1
            best_state = copy.deepcopy(self.model.state_dict())
            no_improve = 0
            print(f'\n  ★ Best composite={composite:.4f} | F1={sf1:.4f} '
                  f'SkillsF1={vskills_f1:.4f} ep={ep} | TrF1={tf1:.4f} LR={lr:.1e}\n')
        else:
            no_improve += 1
            print(f'  ep{ep:>2} vF1={vf1:.4f} SkillsF1={vskills_f1:.4f} '
                  f'tF1={tf1:.4f} P={vp:.3f} R={vr:.3f} pat={no_improve}/{PATIENCE}')
            if no_improve >= PATIENCE:
                print(f'\n  Early stop @ ep{ep}.'); self.control.should_training_stop = True
        return vm


trainer = NERTrainer(
    model=model, args=training_args,
    train_dataset=tok['train'], eval_dataset=tok['val'],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print(f"\n{'='*65}")
print(f"  Train: {len(tok['train']):,} | Val: {len(tok['val'])} ({VAL_RATIO*100:.0f}%) | Test: {len(tok['test'])} ({TEST_RATIO*100:.0f}%)")
print(f"  Model: {MODEL_NAME} + LoRA r={LORA_R} α={LORA_ALPHA}")
print(f"  Early stop: 70% overall F1 + 30% Skills F1")
print(f"{'='*65}\n")

trainer.train()

if best_state:
    model.load_state_dict(best_state)
    print(f'\n  Restored best | F1={best_f1:.4f} | SkillsF1={best_skills_f1:.4f}')


  Train: 1,114 | Val: 33 (15%) | Test: 33 (15%)
  Model: dslim/bert-base-NER + LoRA r=32 α=64
  Early stop: 70% overall F1 + 30% Skills F1



Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,College name F1,Companies worked at F1,Degree F1,Designation F1,Email address F1,Graduation year F1,Location F1,Name F1,Skills F1,Years of experience F1
1,4.962041,2.126495,0.296451,0.508961,0.374670,0.836500,0.051282,0.449612,0.195122,0.353741,0.655738,0.000000,0.359223,0.704225,0.000000,0.000000
2,3.081224,1.474749,0.463736,0.756272,0.574932,0.910626,0.571429,0.544118,0.666667,0.442953,0.776119,0.204082,0.710744,0.969697,0.266667,0.444444
3,2.620688,1.344607,0.528868,0.820789,0.643258,0.928220,0.685714,0.656489,0.705882,0.509804,0.787879,0.250000,0.807339,1.000000,0.375000,0.466667
4,2.545913,1.296061,0.567130,0.878136,0.689170,0.935022,0.750000,0.637037,0.736842,0.680272,0.800000,0.237288,0.876190,1.000000,0.400000,0.482759
5,2.494850,1.267147,0.643766,0.906810,0.752976,0.946986,0.838710,0.711111,0.833333,0.791045,0.787879,0.300000,0.859813,1.000000,0.411765,0.521739
6,2.452072,1.239697,0.719008,0.935484,0.813084,0.962702,0.896552,0.725926,0.882353,0.859375,0.812500,0.480000,0.876190,1.000000,0.687500,0.583333
7,2.417285,1.221884,0.730878,0.924731,0.816456,0.969505,0.800000,0.774194,0.888889,0.854839,0.800000,0.352941,0.921569,1.000000,0.571429,0.875000
8,2.429371,1.215490,0.771261,0.942652,0.848387,0.975135,0.866667,0.746032,0.941176,0.916667,0.800000,0.500000,0.921569,1.000000,0.727273,0.875000
9,2.373074,1.209212,0.780415,0.942652,0.853896,0.976777,0.866667,0.793388,0.914286,0.932203,0.787879,0.500000,0.921569,1.000000,0.628571,0.933333
10,2.366436,1.212066,0.771261,0.942652,0.848387,0.974431,0.866667,0.780488,0.941176,0.901639,0.787879,0.518519,0.921569,1.000000,0.628571,0.933333



  ★ Best composite=0.2623 | F1=0.3747 SkillsF1=0.0000 ep=1 | TrF1=0.3287 LR=2.0e-04


  ★ Best composite=0.4124 | F1=0.4748 SkillsF1=0.2667 ep=2 | TrF1=0.5952 LR=2.0e-04


  ★ Best composite=0.5389 | F1=0.6091 SkillsF1=0.3750 ep=3 | TrF1=0.6669 LR=1.9e-04


  ★ Best composite=0.5864 | F1=0.6662 SkillsF1=0.4000 ep=4 | TrF1=0.6945 LR=1.8e-04


  ★ Best composite=0.6283 | F1=0.7211 SkillsF1=0.4118 ep=5 | TrF1=0.7644 LR=1.6e-04


  ★ Best composite=0.7544 | F1=0.7830 SkillsF1=0.6875 ep=6 | TrF1=0.8314 LR=1.4e-04

  ep 7 vF1=0.8165 SkillsF1=0.5714 tF1=0.8287 P=0.731 R=0.925 pat=1/3

  ★ Best composite=0.8009 | F1=0.8324 SkillsF1=0.7273 ep=8 | TrF1=0.8755 LR=9.3e-05

  ep 9 vF1=0.8539 SkillsF1=0.6286 tF1=0.8822 P=0.780 R=0.943 pat=1/3
  ep10 vF1=0.8484 SkillsF1=0.6286 tF1=0.8855 P=0.771 R=0.943 pat=2/3
  ep11 vF1=0.8506 SkillsF1=0.6286 tF1=0.8986 P=0.777 R=0.939 pat=3/3

  Early stop @ ep11.

  Restored best | F1=0.8324 | SkillsF1=0.7273


In [20]:
# Cell 16: Training Curves
ep = hist['ep']
bi = int(np.argmax(hist['vl_f1'])) if hist['vl_f1'] else 0
be = ep[bi] if ep else 0

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle(
    f'Resume NER v7 | Val={len(val_tokens)}({VAL_RATIO*100:.0f}%) Test={len(test_tokens)}({TEST_RATIO*100:.0f}%) | '
    f'Best F1={best_f1:.4f} SkillsF1={best_skills_f1:.4f}',
    fontsize=11, fontweight='bold'
)

axes[0,0].plot(ep, hist['tr_loss'],'b-o',ms=4,lw=2,label='Train Loss')
axes[0,0].plot(ep, hist['vl_loss'],'r-o',ms=4,lw=2,label='Val Loss')
axes[0,0].axvline(be,color='green',ls='--',lw=1.5,label=f'Best ep={be}')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ep, hist['vl_acc'],'r-o',ms=4,lw=2,label='Val Accuracy')
axes[0,1].axhline(0.95,color='orange',ls='--',alpha=0.7,label='0.95')
axes[0,1].set_ylim(0.5,1.05)
axes[0,1].set_title('Token Accuracy'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

axes[0,2].plot(ep, hist['vl_f1'],'g-o',ms=4,lw=2,label='Val F1')
axes[0,2].plot(ep, hist['tr_f1'],'b--s',ms=3,lw=1.5,label='Train F1',alpha=0.7)
axes[0,2].axhline(best_f1,color='darkgreen',ls='--',lw=1.5,label=f'Best={best_f1:.4f}')
axes[0,2].set_ylim(0,1.05)
axes[0,2].set_title('Overall F1'); axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

axes[1,0].plot(ep, hist['vl_p'],'b-o',ms=4,lw=2,label='Precision')
axes[1,0].plot(ep, hist['vl_r'],'r-o',ms=4,lw=2,label='Recall')
axes[1,0].set_title('Precision vs Recall'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(ep, hist['vl_skills_f1'],'purple',marker='o',ms=4,lw=2,label='Skills F1')
if hist['vl_skills_f1']:
    best_sk = max(hist['vl_skills_f1'])
    axes[1,1].axhline(best_sk,color='purple',ls='--',lw=1.5,label=f'Best={best_sk:.4f}')
axes[1,1].set_ylim(0,1.05)
axes[1,1].set_title('Skills F1'); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

axes[1,2].plot(ep, hist['lr'],'orange',marker='o',ms=4,lw=2,label='LR')
axes[1,2].set_title('Learning Rate'); axes[1,2].legend(); axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves_v7.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {OUTPUT_DIR}/training_curves_v7.png')

Saved: ./resume_ner_v7_lora/training_curves_v7.png


In [21]:
# Cell 17: Evaluation
print(f"\n{'═'*65}\n  VAL SET ({len(val_tokens)} real — {VAL_RATIO*100:.0f}%)\n{'═'*65}")
vr = trainer.evaluate(tok['val'])
print(f"  F1={vr.get('eval_f1',0):.4f}  P={vr.get('eval_precision',0):.4f}  "
      f"R={vr.get('eval_recall',0):.4f}  Acc={vr.get('eval_accuracy',0):.4f}")

print(f"\n{'═'*65}\n  TEST SET ({len(test_tokens)} real — UNSEEN)\n{'═'*65}")
tr_eval = trainer.evaluate(tok['test'])
print(f"  F1={tr_eval.get('eval_f1',0):.4f}  P={tr_eval.get('eval_precision',0):.4f}  "
      f"R={tr_eval.get('eval_recall',0):.4f}  Acc={tr_eval.get('eval_accuracy',0):.4f}")

print('\n  Per-Entity F1 (Test):')
ent_f1 = {k.replace('eval_','').replace('_f1','').replace('_',' ').title(): v
          for k, v in tr_eval.items()
          if '_f1' in k and k != 'eval_f1' and isinstance(v, float)}
for e, f in sorted(ent_f1.items(), key=lambda x: -x[1]):
    bar  = '█' * int(f * 20)
    icon = '✅' if f >= 0.60 else '⚠️'
    print(f'    {e:<28}  {f:.4f}  {bar:<20}  {icon}')


═════════════════════════════════════════════════════════════════
  VAL SET (33 real — 15%)
═════════════════════════════════════════════════════════════════


  F1=0.8484  P=0.7713  R=0.9427  Acc=0.9751

═════════════════════════════════════════════════════════════════
  TEST SET (33 real — UNSEEN)
═════════════════════════════════════════════════════════════════


  F1=0.8368  P=0.7555  R=0.9377  Acc=0.9757

  Per-Entity F1 (Test):
    Name                          1.0000  ████████████████████  ✅
    Designation                   0.9091  ██████████████████    ✅
    Location                      0.9011  ██████████████████    ✅
    Years Of Experience           0.8750  █████████████████     ✅
    Companies Worked At           0.8679  █████████████████     ✅
    Email Address                 0.8060  ████████████████      ✅
    Degree                        0.8000  ████████████████      ✅
    College Name                  0.6857  █████████████         ✅
    Skills                        0.6667  █████████████         ✅
    Graduation Year               0.4783  █████████             ⚠️


In [22]:
# Cell 18: Save Model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(f'{OUTPUT_DIR}/run_meta.json', 'w') as f:
    json.dump({
        'version': 'v7-LoRA',
        'base_model': MODEL_NAME,
        'lora': {'r': LORA_R, 'alpha': LORA_ALPHA, 'modules': LORA_MODULES},
        'split': {'val_ratio': VAL_RATIO, 'test_ratio': TEST_RATIO,
                  'val_size': len(val_tokens), 'test_size': len(test_tokens)},
        'best_val_f1': round(best_f1, 4),
        'best_skills_f1': round(best_skills_f1, 4),
        'test_f1': round(tr_eval.get('eval_f1', 0), 4),
        'improvements_over_v6': [
            'Val/Test 15% each (was 10%)',
            'Synthetic data teaches header-only designation',
            'Post-processing: designation picker, degree header filter + GPA filter',
            'Composite early stopping: 70% F1 + 30% Skills F1',
            'Companies: filter usernames, emails, urls',
        ],
    }, f, indent=2)

print(f'Saved → {OUTPUT_DIR}/')

Saved → ./resume_ner_v7_lora/


In [23]:
# ============================================================
# Cell 19: ✅ Production Inference Engine v7 — FIXED
# إصلاحات:
#   1) Designation → واحد بس (header priority + skills match)
#   2) Degree → يشيل section headers + GPA + date ranges
#   3) Name → dedup + no email
#   4) Location → dedup longest
#   5) YoE → فلتر 12M, 12%, أرقام > 50
#   6) Skills → comma split + fallback
#   7) Companies → شيل usernames و emails و urls
# ============================================================
NOISE_RE   = re.compile(r"^[\s\-|,.:;()\[\]{}\'\"]+|[\s\-|,.:;()\[\]{}\'\"]+$")
_NOISE_TOK = {':',';','-','.','|','/','(',')','+','*','#','@','&','\u2014','\u2013'}
EMAIL_RE   = re.compile(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}')
YOE_NOISE  = re.compile(r'[%MmKkBb+]')
PHONE_RE   = re.compile(r'(?:\+?\d[\d\s\-]{7,}\d)')
GPA_RE     = re.compile(r'\b(gpa|cgpa|grade)\b', re.IGNORECASE)

model.eval()

def tta_predict(text):
    words = re.findall(r'\S+', text)
    if not words: return []
    acc = np.zeros((len(words), N_LABELS), dtype=np.float64)

    for run in range(TTA_RUNS + 1):
        if run > 0: model.train()
        else:       model.eval()
        for i in range(0, len(words), 128):
            chunk = words[i:i+128]
            enc   = tokenizer(
                chunk, is_split_into_words=True, return_tensors='pt',
                truncation=True, max_length=MAX_LEN, padding=False
            )
            with torch.no_grad():
                logits = model(**{k: v.to(DEVICE) for k, v in enc.items()}).logits[0]
            probs = torch.softmax(logits, -1).cpu().numpy()
            seen  = set()
            for ti, wid in enumerate(enc.word_ids(0)):
                if wid is None or wid in seen: continue
                seen.add(wid)
                gw = i + wid
                if gw < len(words): acc[gw] += probs[ti]

    acc /= (TTA_RUNS + 1)
    model.eval()
    return [{'word': words[i], 'label': id2label[int(np.argmax(acc[i]))],
             'score': float(np.max(acc[i]))} for i in range(len(words))]


# ── Helpers ───────────────────────────────────────────────────────
def _is_email(s):      return bool(EMAIL_RE.fullmatch(s.strip()))
def _is_phone(s):      return bool(PHONE_RE.fullmatch(s.strip()))
def _is_url(s):        return any(s.startswith(p) for p in ['http','www.','linkedin','github'])
def _is_section_header(s):
    return s.lower().strip().rstrip(':') in SECTION_HEADERS

def _is_valid_yoe(s):
    s = s.strip()
    if YOE_NOISE.search(s): return False
    m = re.match(r'^\(?(\d{1,2})', s)
    if not m: return False
    return 1 <= int(m.group(1)) <= 50

def _deduplicate_locations(locs):
    if not locs: return locs
    sorted_locs = sorted(set(locs), key=len, reverse=True)
    kept = []
    for loc in sorted_locs:
        if not any(loc.lower() in k.lower() for k in kept):
            kept.append(loc)
    return kept

def _split_comma_skills(skill_str):
    parts = [p.strip().strip('-').strip() for p in re.split(r'[,\u060c]', skill_str)]
    return [p for p in parts if len(p) >= 2]

def _pick_best_designation(candidates, skills):
    if not candidates: return []
    filtered = [c for c in candidates if not _is_section_header(c)]
    if not filtered: return []
    if len(filtered) == 1: return [filtered[0]]

    skill_words = {s.lower() for s in skills}

    def designation_score(d):
        d_lower = d.lower()
        known = any(d_lower == k.lower() for k in DESIGNATION_LIST)
        d_words = set(d_lower.split())
        skill_overlap = len(d_words & skill_words)
        length_penalty = len(d.split())
        return (int(known) * 10 + skill_overlap * 2 - length_penalty)

    best = max(filtered, key=designation_score)
    return [best]

def _pick_best_degree(candidates):
    """
    ✅ يختار degree واحدة:
    - يشيل section headers
    - يشيل أي حاجة فيها GPA / CGPA / Grade
    - يشيل date ranges زي 09/2023 - 05/2027
    - يفضّل الأطول (أكثر تفصيلاً)
    """
    if not candidates: return []
    filtered = [c for c in candidates
                if not _is_section_header(c)
                and not _is_email(c)
                and not _is_phone(c)
                and not GPA_RE.search(c)
                and not re.search(r'\d{4}.*\d{4}', c)
                and len(c.split()) >= 2]
    if not filtered: return []
    return [max(filtered, key=len)]

def skills_fallback(text, found_skills):
    text_lower  = text.lower()
    found_lower = {s.lower() for s in found_skills}
    extra = []
    for sk in sorted(ALL_SKILLS_FLAT, key=len, reverse=True):
        sk_lower = sk.lower()
        if sk_lower in found_lower: continue
        if re.search(r'\b' + re.escape(sk_lower) + r'\b', text_lower):
            extra.append(sk)
            found_lower.add(sk_lower)
    return extra


def group_entities(preds, text, conf=INF_CONF):
    ents, cur_type, cur_words = {}, None, []

    def flush():
        if cur_type and cur_words:
            v = NOISE_RE.sub('', ' '.join(cur_words)).strip()
            if len(v) >= 2:
                ents.setdefault(cur_type, []).append(v)

    for wp in preds:
        lbl, sc, w = wp['label'], wp['score'], wp['word']
        if w in _NOISE_TOK: flush(); cur_type, cur_words = None, []; continue
        if lbl.startswith('B-') and sc >= conf:
            flush(); cur_type, cur_words = lbl[2:], [w]
        elif lbl.startswith('I-') and cur_type == lbl[2:] and sc >= conf * 0.75:
            cur_words.append(w)
        else:
            flush(); cur_type, cur_words = None, []
    flush()

    # ── Email fallback ───────────────────────────────────────────
    emails = [e for e in EMAIL_RE.findall(text) if not _is_phone(e)]
    if emails: ents['Email Address'] = list(dict.fromkeys(emails))

    # ── ✅ Companies: شيل emails, usernames, urls ────────────────
    comp_key = next((k for k in ents if 'compan' in k.lower() or 'employer' in k.lower()), None)
    if comp_key:
        ents[comp_key] = [
            c for c in ents[comp_key]
            if not _is_email(c)
            and '@' not in c
            and not _is_url(c)
            and not re.fullmatch(r'[a-zA-Z0-9._+\-]+', c)
        ]
        if not ents[comp_key]: del ents[comp_key]

    # ── Name: شيل emails, phones, urls ──────────────────────────
    if 'Name' in ents:
        ents['Name'] = [n for n in ents['Name']
                        if not _is_email(n) and not _is_phone(n) and not _is_url(n)
                        and len(n.split()) >= 1]
        if ents['Name']:
            names_s = sorted(set(ents['Name']), key=len)
            kept = []
            for nm in names_s:
                if not any(nm.lower() in k.lower() for k in kept):
                    kept.append(nm)
            ents['Name'] = [kept[0]] if kept else []
        if not ents['Name']: del ents['Name']

    # ── Location: dedup ──────────────────────────────────────────
    if 'Location' in ents:
        ents['Location'] = _deduplicate_locations(ents['Location'])

    # ── YoE: فلتر ────────────────────────────────────────────────
    yoe_key = next((k for k in ents if 'year' in k.lower() or 'experience' in k.lower()), None)
    if yoe_key:
        ents[yoe_key] = [y for y in ents[yoe_key] if _is_valid_yoe(y)]
        if not ents[yoe_key]: del ents[yoe_key]

    # ── Skills: comma split + dedup + fallback ───────────────────
    raw_skills = ents.get('Skills', [])
    expanded = []
    for sk in raw_skills:
        if ',' in sk: expanded.extend(_split_comma_skills(sk))
        else:          expanded.append(sk.strip('-').strip())
    expanded = [s for s in expanded
                if len(s) >= 2 and not _is_section_header(s)
                and not s.lower() in {'skills','technical','soft','tools'}]
    extra = skills_fallback(text, expanded)
    all_skills = list(dict.fromkeys(expanded + extra))
    if all_skills: ents['Skills'] = all_skills
    elif 'Skills' in ents: del ents['Skills']

    # ── ✅ Designation: pick best one ────────────────────────────
    if 'Designation' in ents:
        current_skills = ents.get('Skills', [])
        ents['Designation'] = _pick_best_designation(ents['Designation'], current_skills)
        if not ents['Designation']: del ents['Designation']

    # ── ✅ Degree: pick best one ─────────────────────────────────
    if 'Degree' in ents:
        ents['Degree'] = _pick_best_degree(ents['Degree'])
        if not ents['Degree']: del ents['Degree']

    return ents


def pretty_print(label, entities):
    ORDER = ['Name','Designation','Email Address','Location',
             'Companies worked at','Degree','College Name',
             'Graduation Year','Years of Experience','Skills']
    print(f"\n  {label}")
    print(f"  {'─'*64}")
    printed = set()
    for et in ORDER:
        if et in entities:
            printed.add(et)
            for i, v in enumerate(entities[et]):
                tag = et if i == 0 else ''
                print(f"  {tag:<30}  {v[:60]}")
    for et, vals in entities.items():
        if et not in printed:
            for i, v in enumerate(vals):
                tag = et if i == 0 else ''
                print(f"  {tag:<30}  {v[:60]}")
    print()

In [24]:
# Cell 20: Text Inference Test
TEST_RESUMES = [
    {'label': 'Data Analyst (header test)', 'text': (
        'Mohamed Hammad\n'
        'Data Analyst\n'
        'Mohamedeeexx995@gmail.com | Cairo, Egypt\n\n'
        'EDUCATION\n'
        "Bachelor's in Artificial Intelligence\n"
        'Helwan University | 2027\n\n'
        'EXPERIENCE\n'
        'Data Analyst | Microsoft Power BI Specialist, DEPI\n'
        '06/2025 – 12/2025 | Cairo, Egypt\n\n'
        'SKILLS\n'
        '- Python\n- SQL\n- Power BI\n- Tableau\n- Excel\n'
        '- Machine Learning\n- Feature Engineering\n- Pandas\n- NumPy')},
    {'label': 'ML Engineer', 'text': (
        'Ahmed Ali\nMachine Learning Engineer\n'
        'ahmed.ali@gmail.com | Cairo\n\n'
        'EDUCATION\nM.Sc Artificial Intelligence\n'
        'Cairo University | 2022\n\n'
        'EXPERIENCE\nGoogle - ML Engineer (2022 - Present)\n\n'
        'SKILLS\n- TensorFlow\n- PyTorch\n- Python\n- Docker\n- AWS')},
]

print(f"\n{'='*65}")
print(f"  TEXT INFERENCE | TTA {TTA_RUNS+1} passes | conf>={INF_CONF}")
print(f"{'='*65}")
for r in TEST_RESUMES:
    out = group_entities(tta_predict(r['text']), r['text'])
    pretty_print(r['label'], out)


  TEXT INFERENCE | TTA 4 passes | conf>=0.5

  Data Analyst (header test)
  ────────────────────────────────────────────────────────────────
  Name                            Mohamed Hammad
  Designation                     Data Analyst
  Email Address                   Mohamedeeexx995@gmail.com
  Degree                          Bachelor's in Artificial Intelligence
  College Name                    Helwan University
  Skills                          Python
                                  SQL
                                  Power BI
                                  Tableau
                                  Excel
                                  Machine Learning
                                  Feature Engineering
                                  Pandas
                                  NumPy


  ML Engineer
  ────────────────────────────────────────────────────────────────
  Name                            Ahmed Ali
  Designation                     Machine Learning Engineer
 

In [32]:
# Cell 21: PDF Inference
def run_pdf_inference(pdf_path: str):
    print(f"\n{'='*65}")
    print(f"  PDF INFERENCE: {pdf_path}")
    print(f"{'='*65}")

    reader    = PdfReader(pdf_path)
    full_text = ''
    for page in reader.pages:
        t = page.extract_text()
        if t: full_text += t + '\n'

    print(f"  {len(reader.pages)} page(s) | {len(full_text):,} chars")

    if not full_text.strip():
        print("  ⚠️  No extractable text (scanned PDF?)"); return

    preds = tta_predict(full_text)
    ents  = group_entities(preds, full_text)
    pretty_print(f'Full Document ({len(reader.pages)} pages)', ents)
    print(f"{'='*65}\n")
    return ents


PDF_PATH = '/content/Mohamed_Ahmed_CV.pdf'

import os
if os.path.exists(PDF_PATH):
    run_pdf_inference(PDF_PATH)
else:
    print(f"⚠️  File not found: {PDF_PATH}")
    print("   غيّر PDF_PATH للمسار الصح\n")

    # Demo simulation لـ CV محمد حماد
    print("  ─── Demo: CV محمد حماد ───")
    demo_text = """Mohamed Hammad
Data Analyst
Mohamedeeexx995@gmail.com +201118319050 Cairo, Egypt
linkedin.com/in/mohamed-ahmed-289b9a2b8 github.com/kmtrrdev

SUMMARY
Data Analyst and Machine Learning Engineer specializing in transforming raw data.

EDUCATION
Bachelor's in Artificial Intelligence
Helwan University
GPA: 3.5 | 09/2023 – 05/2027 | Cairo, Egypt

EXPERIENCE
Data Analyst | Microsoft Power BI Specialist, DEPI
06/2025 – 12/2025 | Cairo, Egypt
• Developed interactive dashboards using Power BI and Tableau.
• Processed data using Power Query and SQL.

SKILLS
Technical Skills: Python , SQL , Power BI , Tableau , Excel , Data Cleaning , Data Transformation , Data Modeling
Machine Learning, Feature Engineering , Model Evaluation , Pandas, NumPy , Scikit-learn
Soft Skills: Analytical Thinking , Problem Solving , Communication
"""
    preds = tta_predict(demo_text)
    ents  = group_entities(preds, demo_text)
    pretty_print('Mohamed Hammad CV (Demo)', ents)


  PDF INFERENCE: /content/Mohamed_Ahmed_CV.pdf
  2 page(s) | 1,874 chars

  Full Document (2 pages)
  ────────────────────────────────────────────────────────────────
  Name                            Msp Front-End
  Designation                     Contact information
  Email Address                   Mohammed_20230449@fci.helwan.edu.eg
  Degree                          Bachelor's in Computer Science and Artificial Intelligence
  College Name                    Helwan University
  Skills                          Html
                                  JavaScript
                                  Bootstrap
                                  GitHub
                                  React
                                  Linux
                                  Java
                                  CSS
                                  SQL
                                  C


